In [1]:
import os
import cv2
import yaml
from pathlib import Path
from ultralytics import YOLO

def executar_pipeline_completo():
    print("="*70)
    print("PIPELINE YOLO: AUTO-LABELING FOCADO EM CLASSE ÚNICA (ABACAXI)")
    print("="*70)

    # =================================================================
    # 1. CRIAÇÃO DAS PASTAS
    # =================================================================
    base_dir = Path.cwd() / "meu_dataset"
    pasta_img_treino = base_dir / "images" / "train"
    pasta_lbl_treino = base_dir / "labels" / "train"

    pasta_img_treino.mkdir(parents=True, exist_ok=True)
    pasta_lbl_treino.mkdir(parents=True, exist_ok=True)

    print(f"\n[PASSO 1] Pastas criadas em: {base_dir}")
    print(f"-> VÁ ATÉ A PASTA: {pasta_img_treino}")
    print("-> Cole lá as imagens do Abacaxi.")
    
    input("\n[AÇÃO] Pressione ENTER no terminal quando as imagens estiverem na pasta...")

    # =================================================================
    # 2. AUTO-LABELING COM FILTRO DE CLASSE (ID Remapping)
    # =================================================================
    imagens = list(pasta_img_treino.glob("*.jpg")) + list(pasta_img_treino.glob("*.png")) + list(pasta_img_treino.glob("*.jpeg"))
    
    if len(imagens) == 0:
        print("[ERRO] Nenhuma imagem encontrada. Encerrando script.")
        return

    print(f"\n[PASSO 2] Iniciando Auto-Labeling Inteligente em {len(imagens)} imagens...")
    modelo_professor = YOLO('yolov8x-oiv7.pt')
    
    # Busca dinamicamente qual é o ID da classe "Pineapple" dentro do modelo OIV7
    id_alvo_professor = None
    for k, v in modelo_professor.names.items():
        if v.lower() == 'pineapple':
            id_alvo_professor = k
            break
            
    if id_alvo_professor is None:
        print("[ERRO] A classe 'Pineapple' não existe no modelo OIV7. Verifique o nome.")
        return

    print(f"-> Classe 'Pineapple' encontrada no professor com o ID: {id_alvo_professor}")
    print(f"-> Remapeando para o ID: 0 (Abacaxi) no nosso novo dataset!")

    imagens_removidas = 0
    anotacoes_geradas = 0

    for img_path in imagens:
        teste_img = cv2.imread(str(img_path))
        if teste_img is None:
            print(f"[AUTO-CURA] Arquivo ilegível expurgado: {img_path.name}")
            os.remove(img_path)
            imagens_removidas += 1
            continue

        resultados = modelo_professor.predict(source=str(img_path), conf=0.40, verbose=False)
        
        if not resultados or len(resultados) == 0:
            continue
        
        txt_path = pasta_lbl_treino / f"{img_path.stem}.txt"
        
        # Conta quantas caixas de abacaxi achou nesta imagem
        abacaxis_na_foto = 0
        
        with open(txt_path, 'w') as f:
            for box in resultados[0].boxes:
                cls_id = int(box.cls[0].item())
                
                # O PULO DO GATO: Só salva se for o abacaxi!
                if cls_id == id_alvo_professor:
                    x, y, w, h = box.xywhn[0].tolist() 
                    # Grava forçadamente como classe 0
                    f.write(f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")
                    abacaxis_na_foto += 1
                    anotacoes_geradas += 1
        
        # Se não achou nenhum abacaxi na foto, apaga o arquivo txt vazio para não sujar o treino
        if abacaxis_na_foto == 0:
            os.remove(txt_path)

    print(f"[OK] Auto-Labeling concluído! {anotacoes_geradas} abacaxis anotados.")

    # =================================================================
    # 3. GERANDO O YAML DE CLASSE ÚNICA E TREINANDO
    # =================================================================
    yaml_path = base_dir / "dataset.yaml"
    config = {
        'path': str(base_dir.absolute()),
        'train': 'images/train',
        'val': 'images/train', 
        'nc': 1,  
        'names': {0: 'abacaxi'} 
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, sort_keys=False)

    print("\n[PASSO 3] Iniciando o Treinamento do Modelo (Single Class)...")
    modelo_aluno = YOLO('yolov8n.yaml')
    modelo_aluno.train(data=str(yaml_path), epochs=5, imgsz=640)
    print("[OK] Treinamento concluído!")

    # =================================================================
    # 4. TESTE FINAL
    # =================================================================
    print("\n[PASSO 4] Testando o modelo treinado!")
    
    # ATENÇÃO: Verifique se o caminho da imagem de teste está correto no seu PC
    caminho_teste = r'C:\Users\welin\OneDrive\Área de Trabalho\testes 365\PAG-GOV-ABACAXI-NORTE-1.jpg'

    if Path(caminho_teste).exists():
        print(f"\n-> Analisando imagem de teste: {Path(caminho_teste).name}...")
        
        # Reduzi um pouco a confiança para garantir que ele mostre mesmo se estiver incerto
        resultados_teste = modelo_aluno.predict(source=caminho_teste, conf=0.15, verbose=False)
        
        if not resultados_teste or len(resultados_teste) == 0:
            print("[ERRO] Falha ao ler a imagem de teste.")
            return

        resultado = resultados_teste[0]
        
        print("\n" + "="*50)
        print("  RESULTADO DA DETECÇÃO (MOTOR DE INFERÊNCIA)  ")
        print("="*50)
        
        total_objetos = len(resultado.boxes)
        
        if total_objetos == 0:
            print("-> A IA não encontrou nenhum objeto.")
        else:
            print(f"-> A IA localizou {total_objetos} objeto(s) na imagem:\n")
            for i, box in enumerate(resultado.boxes):
                cls_id = int(box.cls[0].item())
                nome_classe = modelo_aluno.names[cls_id]
                confianca = float(box.conf[0].item()) * 100
                print(f"   [Alvo {i+1}] Classe: '{nome_classe}' | Confiança: {confianca:.1f}%")
        print("="*50)

        arquivo_saida = "resultado_teste_abacaxi.jpg"
        resultado.save(filename=arquivo_saida)
        print(f"\n[SISTEMA] Imagem salva em: {arquivo_saida}")
        resultado.show() 
    else:
        print(f"[ERRO] Imagem de teste não encontrada: {caminho_teste}")

if __name__ == "__main__":
    executar_pipeline_completo()

PIPELINE YOLO: AUTO-LABELING FOCADO EM CLASSE ÚNICA (ABACAXI)

[PASSO 1] Pastas criadas em: c:\Users\welin\OneDrive\Área de Trabalho\testes 365\meu_dataset
-> VÁ ATÉ A PASTA: c:\Users\welin\OneDrive\Área de Trabalho\testes 365\meu_dataset\images\train
-> Cole lá as imagens do Abacaxi.

[PASSO 2] Iniciando Auto-Labeling Inteligente em 20 imagens...
-> Classe 'Pineapple' encontrada no professor com o ID: 389
-> Remapeando para o ID: 0 (Abacaxi) no nosso novo dataset!
[OK] Auto-Labeling concluído! 38 abacaxis anotados.

[PASSO 3] Iniciando o Treinamento do Modelo (Single Class)...
New https://pypi.org/project/ultralytics/8.4.155 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.153  Python-3.14.4 torch-2.14.0+cpu CPU (11th Gen Intel Core i5-11400H @ 2.70GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=